## Import the datasets

In [5]:
import pandas as pd
from pathlib import Path

DATA_FOLDERPATH = Path("src/data")

POLICIES_AND_CLAIMS_PATH = DATA_FOLDERPATH / "policies_and_claims.parquet"
INDUSTRY_DATA_PATH = DATA_FOLDERPATH / "uk_industry_wide_annual_losses.parquet"

df_policies_and_claims = pd.read_parquet(POLICIES_AND_CLAIMS_PATH)
df_industry_data = pd.read_parquet(INDUSTRY_DATA_PATH)

## Understand ```policy_and_claims.parquet```

In [ ]:
df_policies_and_claims.head(5)

,policy_id,year,sum_insured,excess,geographic_flood_risk,premium,claim_count,claim_cost
0,1000001,2025,166000,250,1. Very Low,31.82,0,0.0
1,1000002,2023,397000,200,1. Very Low,125.77,0,0.0
2,1000003,2025,244000,200,1. Very Low,51.02,0,0.0
3,1000004,2024,327000,100,2. Low,107.32,0,0.0
4,1000005,2021,215000,200,2. Low,76.63,0,0.0


In [ ]:
df_policies_and_claims.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 10030123 entries, 0 to 10030122
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   policy_id              int64  
 1   year                   int64  
 2   sum_insured            int64  
 3   excess                 int64  
 4   geographic_flood_risk  str    
 5   premium                float64
 6   claim_count            int64  
 7   claim_cost             float64
dtypes: float64(2), int64(5), str(1)
memory usage: 692.5 MB


**Column meanings:**
- **policy_id**: Unique ID of a Policy (not important for this exercise)
- **year**: The calendar year the policy was written (we may wish to look at loss ratio by year)
- **sum_insured**: The estimated rebuilding cost of the house covered by the policy
- **excess**: The excess for the policy
- **geographic_flood_risk**: The level of flood risk at the exact address, using our in-house flood risk map
- **premium**: The flood premium for the policy
- **claim_count**: 1 if there was a claim, 0 otherwise
- **claim_cost**: The cost of claims for that policy

A quick look at loss ratio by year:

In [ ]:
(
    df_policies_and_claims.groupby("year")
    .agg(
        {
            "sum_insured": "sum",
            "premium": "sum",
            "claim_count": "sum",
            "claim_cost": "sum",
        }
    )
    .reset_index()
    .assign(loss_ratio=lambda x: x["claim_cost"] / x["premium"])
    .assign(claim_cost_per_sum_insured=lambda x: x["claim_cost"] / x["sum_insured"])
)

,year,sum_insured,premium,claim_count,claim_cost,loss_ratio,claim_cost_per_sum_insured
0,2021,458444746000,1.569565e+08,708,34072624.84,0.217083,0.000074
1,2022,431669482000,1.477878e+08,115,5905715.57,0.039961,0.000014
2,2023,431260295000,1.476504e+08,408,19091909.40,0.129305,0.000044
3,2024,565364209000,1.662070e+08,1416,73326608.44,0.441176,0.000130
4,2025,808339234000,1.872287e+08,178,9611663.81,0.051336,0.000012


## Understand ```uk_industry_wide_annual_losses.parquet```

In [9]:
df_industry_data.head(5)

,year,uk_count_of_houses,uk_count_of_houses_flooded
0,2000,20423423,35714
1,2001,20525540,1026
2,2002,20628168,5157
3,2003,20731309,4146
4,2004,20834965,54052


In [10]:
df_industry_data.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   year                        26 non-null     int64
 1   uk_count_of_houses          26 non-null     int64
 2   uk_count_of_houses_flooded  26 non-null     int64
dtypes: int64(3)
memory usage: 756.0 bytes


Look at flood frequency per house per year:

In [11]:
(
    df_industry_data.groupby("year")
    .sum()
    .reset_index()
    .assign(
        uk_flood_frequency_per_house=lambda x: (
            x["uk_count_of_houses_flooded"] / x["uk_count_of_houses"]
        )
    )
)

,year,uk_count_of_houses,uk_count_of_houses_flooded,uk_flood_frequency_per_house
0,2000,20423423,35714,0.001749
1,2001,20525540,1026,0.000050
2,2002,20628168,5157,0.000250
3,2003,20731309,4146,0.000200
4,2004,20834965,54052,0.002594
5,2005,20939140,2094,0.000100
6,2006,21043836,1052,0.000050
7,2007,21149055,107752,0.005095
8,2008,21254800,3188,0.000150
9,2009,21361074,2136,0.000100
